## Load Data & Perform Initial Inspection

In [ ]:
import pandas as pd

df = pd.read_csv("/workspaces/final-project-abshire/data/pitch_2024_relevant.csv")

# Basic checks
print(df.shape)       # rows, columns
print(df.columns)     # column names
df.head()             # first 5 rows
df.info()             # datatypes & null counts
df.describe()         # summary stats for numeric columns

# Check for duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")
# 0 duplicate rows, that's huge

# Percentage of missing values per column
missing_percent = df.isnull().mean().sort_values(ascending=False) * 100
print(missing_percent)

# Unique counts for all object columns
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].nunique()} unique values")

### Key Column Definitions

- **`events`** → Scorekeeper outcome of the plate appearance.
    - Examples: `single`, `walk`, `double`, `home_run`, `triple`, or `NaN` if nothing happened (e.g., still in the at-bat, or an out in the field without a hit).

- **`description`** → Pitch-by-pitch detail.
    - Examples: `ball`, `foul`, `swinging_strike`, `hit_into_play`.
    - More granular than `events`; describes *what happened on that pitch*.

- **`launch_speed`** – Exit velocity of the batted ball (mph). For untracked balls, Statcast estimates are included. 

- **`launch_angle`** – Launch angle of the batted ball (degrees).

- **`zone`** – Strike zone location number when the ball crosses the plate (1-9 strikes, 11,12,13,14 out of zone).

- **`batter`** – MLB Player ID tied to the play event.

- **`stand`** – Side of the plate the batter is standing on (`L` or `R`). 

- **`p_throws`** – Hand the pitcher throws with (`L` or `R`).
  
**`game_year`** – Year the game took place.  
**`game_date`** – Calendar date of the game (YYYY-MM-DD).  
**`home_team`** – Abbreviation of the home team.    
**`pitch_type`** – Pitch type derived from Statcast classification.  
**`effective_speed`** – Derived speed accounting for pitcher release extension.  
**`pfx_x`** – Horizontal pitch movement in feet (catcher’s perspective).  
**`pfx_z`** – Vertical pitch movement in feet (catcher’s perspective).  
**`plate_x`** – Horizontal location of the ball crossing home plate (catcher’s perspective).  
**`plate_z`** – Vertical location of the ball crossing home plate (catcher’s perspective).      
**`hc_x`** – Hit coordinate X of the batted ball (on-field location).  
**`hc_y`** – Hit coordinate Y of the batted ball (on-field location).  
**`if_fielding_alignment`** – Infield defensive alignment at the time of the pitch.

### Comparing 'description' (pitch-level) and 'events' (outcome-level)

In [ ]:
# Target variable could be a few things... 
# Let's start with looking at description and events

print("Description value counts:")
print(df['description'].value_counts(dropna=False))

print("\nEvents value counts:")
print(df['events'].value_counts(dropna=False))

### Seeing all unique pitch types

In [ ]:
df['pitch_type'].value_counts(dropna=False)

### Isolating Hit Events & Verifying Launch Metrics

In [ ]:
# Unique events
print(df['events'].dropna().unique())

# Make a df with only hits!
hitting_events = ['single', 'double', 'triple', 'home_run']
df_hits = df[df['events'].isin(hitting_events)].copy()
print(f"Filtered to {len(df_hits)} rows with hit events")

# Check if there are any 0's in the luanch speed/angle
df_hits[['launch_speed', 'launch_angle']].isnull().mean() * 100


### Launch Speed & Angle Patterns by Hit Type

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Summary stats of launch speed and launch angle by hit type
summary_stats = df_hits.groupby('events')[['launch_speed', 'launch_angle']].describe()
print(summary_stats)

# Plot distribution of launch speed by hit type
plt.figure(figsize=(10, 6))
sns.histplot(data=df_hits, x='launch_speed', hue='events', bins=30, kde=True, palette='viridis')
plt.title('Launch Speed Distribution by Hit Type')
plt.xlabel('Launch Speed (mph)')
plt.ylabel('Count')
plt.show()

# Scatterplot of launch speed vs launch angle by hit type
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_hits, x='launch_speed', y='launch_angle', hue='events', alpha=0.6, palette='viridis')
plt.title('Launch Speed vs Launch Angle by Hit Type')
plt.xlabel('Launch Speed (mph)')
plt.ylabel('Launch Angle (degrees)')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

# Scatterplot of launch speed vs launch angle by hit type
sns.scatterplot(
    data=df_hits, 
    x='launch_speed', 
    y='launch_angle', 
    hue='events', 
    alpha=0.6, 
    palette='viridis',
    edgecolor=None,
    s=30
)

# Add shaded boxes for typical launch windows (approximate)

# Singles
plt.axvspan(80.4, 101.4, ymin=(1 + 90)/180, ymax=(15 + 90)/180, color='blue', alpha=0.6, label='Typical Single Window')

# Doubles
plt.axvspan(93.7, 104.7, ymin=(12 + 90)/180, ymax=(22 + 90)/180, color='green', alpha=0.6, label='Typical Double Window')

# Triples
plt.axvspan(94.8, 103.6, ymin=(13.75 + 90)/180, ymax=(25 + 90)/180, color='orange', alpha=0.6, label='Typical Triple Window')

# Home Runs
plt.axvspan(101.5, 107.2, ymin=(25 + 90)/180, ymax=(32 + 90)/180, color='red', alpha=0.6, label='Typical HR Window')

plt.title('Launch Speed vs Launch Angle by Hit Type (2024)')
plt.xlabel('Launch Speed (mph)')
plt.ylabel('Launch Angle (degrees)')
plt.legend(title='Hit Type')
plt.grid(True)
plt.show()


### Launch Metrics Summary by Statcast Zone

In [ ]:
print(df['zone'].dropna().unique())

zone_summary = (
    df_hits
    .groupby('zone')[['launch_speed']]
    .agg(['count', 'mean', 'std', 'median'])
    .round(2)
)

zone_summary


### Launch Speed and Angle Distributions by Zone

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot for launch speed by zone
sns.boxplot(data=df_hits, x='zone', y='launch_speed', ax=axes[0], palette='viridis')
axes[0].set_title('Launch Speed by Zone')
axes[0].set_xlabel('Pitch Zone')
axes[0].set_ylabel('Launch Speed (mph)')

# Boxplot for launch angle by zone
sns.boxplot(data=df_hits, x='zone', y='launch_angle', ax=axes[1], palette='magma')
axes[1].set_title('Launch Angle by Zone')
axes[1].set_xlabel('Pitch Zone')
axes[1].set_ylabel('Launch Angle (°)')

plt.tight_layout()
plt.show()


### Heatmap of Mean Launch Speed by Statcast Zone

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Example: your summarized data in a DataFrame
zone_data = {
    'zone': [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14],
    'mean_launch_speed': [91.47, 96.40, 92.19, 94.03, 98.06, 94.45, 94.37, 97.40, 92.67, 84.26, 85.29, 85.45, 83.33]
}
df_zone = pd.DataFrame(zone_data)

# Strike zone layout mapping
zone_grid = pd.DataFrame([
    [11, None, 12],
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
    [13, None, 14] 
])

# Map mean launch speed to the grid
heatmap_values = zone_grid.replace(
    {z: df_zone.set_index('zone')['mean_launch_speed'].get(z) for z in df_zone['zone']}
)

# Plot
plt.figure(figsize=(6,8))
sns.heatmap(
    heatmap_values.astype(float), 
    annot=True, fmt=".1f", cmap="RdYlGn", 
    linewidths=1, cbar_kws={'label': 'Mean Launch Speed (mph)'},
    vmin=80, vmax=100
)
plt.title('Mean Launch Speed by Statcast Zone', fontsize=14)
plt.ylabel('Pitch Height in Zone')
plt.xlabel('Horizontal Location in Zone')
plt.gca().invert_yaxis()
plt.show()

### Hit Outcome Rates by Zone (Stacked Bar Chart)

In [ ]:
# Count total hits per zone
total_per_zone = df_hits.groupby('zone').size()

# Count each hit type per zone
hits_per_zone = df_hits.groupby(['zone', 'events']).size().unstack(fill_value=0)

# Convert to percentage of hits in that zone
hit_rates = hits_per_zone.div(total_per_zone, axis=0) * 100

# Optional: order the columns consistently
hit_rates = hit_rates[['single', 'double', 'triple', 'home_run']]

print(hit_rates.round(2))

# Plot stacked bar chart
hit_rates.plot(kind='bar', stacked=True, figsize=(10,6))
plt.ylabel("Percentage of Hits")
plt.title("Hit Outcome Rates by Zone")
plt.legend(title="Hit Type")
plt.show()


## Transformation + Additional Visualization

In [ ]:
import pandas as pd

# Load data
pitch_df = pd.read_csv('/workspaces/final-project-abshire/data/pitch_2024_relevant.csv')
player_ids_df = pd.read_csv('/workspaces/final-project-abshire/data/player_ids.csv')

# Keep only MLBAMID and Name, rename Name to batter_name
player_ids_sub = player_ids_df[['MLBAMID', 'Name']].rename(columns={'Name': 'batter_name'})

# Merge on batter and MLBAMID
df_hits = pitch_df.merge(player_ids_sub, left_on='batter', right_on='MLBAMID', how='left')

# Drop MLBAMID column if not needed
df_hits = df_hits.drop(columns=['MLBAMID'])

# Rename player_name to pitcher_name
df_hits = df_hits.rename(columns={'player_name': 'pitcher_name'})

# Now define your pitch_type_map
pitch_type_map = {
    'FF': 'Fastball', # Four-Seam
    'FA': 'Fastball', # General Fastball
    'FT': 'Fastball',  # Two-Seam
    'SI': 'Fastball', # Sinker
    'FS': 'Fastball', # Split Finger
    'SL': 'Slider', # Slider
    'SV': 'Slider', # Slurve
    'ST': 'Curveball', # Sweeper
    'CU': 'Curveball', # Curveball
    'KC': 'Curveball', # Knuckle Curve
    'CH': 'Changeup', # Change Up
    'FO': 'Changeup', # Fork Ball
    'EP': 'Changeup', # Eephus
    'FC': 'Fastball', # Cutter
    'KN': 'Knuckleball', # Knuckleball
    'SC': 'Other', # Screwball
    'CS': 'Other', # Slow Curve
    'PO': 'Other', # Pitch Out?
    None: 'Unknown',  # or np.nan mapped to 'Unknown'
    'NaN': 'Unknown'  # if string 'NaN'
}

# Then map pitch_type to pitch_group column on the merged df_hits
df_hits['pitch_group'] = df_hits['pitch_type'].map(pitch_type_map).fillna('Unknown')

print(df_hits.head())


### Simplifying Pitch Types into Pitch Groups (main_pitch_groups) / (pitch_group)

In [ ]:
mean_launch_speed_fastball_zone5 = df_hits[
    (df_hits['pitch_group'] == 'Fastball') & (df_hits['zone'] == 5)
]['launch_speed'].mean()



### Creating Focused Dataset with Binary Hit Outcome Features

In [ ]:
import pandas as pd

# Load data
pitch_df = pd.read_csv('/workspaces/final-project-abshire/data/pitch_2024_relevant.csv')
player_ids_df = pd.read_csv('/workspaces/final-project-abshire/data/player_ids.csv')

# Keep only MLBAMID and Name, rename Name to batter_name
player_ids_sub = player_ids_df[['MLBAMID', 'Name']].rename(columns={'Name': 'batter_name'})

# Merge on batter and MLBAMID
df_hits = pitch_df.merge(player_ids_sub, left_on='batter', right_on='MLBAMID', how='left')

# Drop MLBAMID column if not needed
df_hits = df_hits.drop(columns=['MLBAMID'])

df_hits = df_hits.rename(columns={'player_name': 'pitcher_name'})

print(df_hits.head())

In [ ]:
# List of desired columns
columns_to_keep = [
    'batter_name', "pitcher_name", 'stand', 'p_throws', 'pitch_group', 'effective_speed',
    'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'zone',
    'launch_speed', 'launch_angle', 'hc_x', 'hc_y', 'events', "result"
]
# Create new DataFrame selecting those columns
newdf = df_hits[columns_to_keep].copy()


for hit_type in ['single', 'double', 'triple', 'home_run']:
    newdf[f'is_{hit_type}'] = (newdf['events'] == hit_type).astype(int)

# If batter stands same side as pitcher's throwing hand, else 0
newdf['same_side'] = (newdf['stand'].str.upper() == newdf['p_throws'].str.upper()).astype(int)

# Check the first few rows to confirm
print(newdf.head())